# End of week 1 exercise

This exercise use local ollama models. If you want to use different models, update the models dictionary.

You will need to have the models pulled locally to use them.

There are two steps to this exercise. 
1. Use an LLM call to determine the best model to answer the question.
2. Make the LLM call to answer the question using the model from step 1

In [ ]:
# imports
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [ ]:
# constants

models = {
    "llama3.2:latest": "Simple questions and answers",
    "qwen3:7b":"Questions and answers with moderate complexity",
    "qwen2.5-coder:14b":"Questions related to code and code generation",
    "qwen3:14b":"Complex questions and detailed answers"
}

initial_model = "qwen3:14b"

In [ ]:
# set up environment
openai = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1"  )

system_prompt = """You are an assistant that helps choose the best model for a given question. 
You will be given list of models and the scenarios they are best for. You will also be given question.
Choose which model will be best for answering the question. Respond with only the name of the suggested model.
"""

def list_models():
    model_list = []
    for model_name, description in models.items():
        model_list.append(f"Model Name: {model_name}\nScenario: {description}")
    
    return "\n\n".join(model_list)

def create_user_prompt(question):
    user_prompt = f"""
Here are the available models and the scenarios they are best for:

{list_models()}

Question:
{question}
    """
    return user_prompt

In [ ]:
print(create_user_prompt("Who was Abraham Lincoln?"))

In [ ]:
def choose_model(question):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": create_user_prompt(question)},
    ]
    response = openai.chat.completions.create(model=initial_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
choose_model("Who was Abraham Lincoln?")

This is the cell that pulls everything together to answer the question.

In [ ]:
question = r"""
Who was Abraham Lincoln?
"""

system_prompt2 = """You are an are helpful assistant that answers questions factually.
If you don't know the answer to a question, say you don't know. 
Do not try to make up an answer.

Respond in Markdown
"""

messages = [
    {"role": "system", "content": system_prompt2},
    {"role": "user", "content": question},
]

model = choose_model(question)
print(f"Using model: {model}")
stream = openai.chat.completions.create(model=model, 
                                        messages=messages, 
                                        stream=True)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

